# Generate held-out answers from the fine-tuned model and the untuned base model

This notebook does the one thing a laptop is too slow for: it asks two models to brief all 45 held-out packets.

- the **untuned base model**, Qwen2.5-3B-Instruct, straight from the hub: the "before" column;
- the **fine-tuned student**: the same base model plus the LoRA adapter saved by `01_sft_colab.ipynb` in `MyDrive/news-briefer/news-briefer-lora`.

It writes `base_answers.jsonl` and `student_answers.jsonl` to `MyDrive/news-briefer/`. Scoring happens back in the repo:

    PYTHONPATH=src:. .venv/bin/python training/evaluate_student.py --answers base_answers.jsonl --label base --out training/data/base_eval.json
    PYTHONPATH=src:. .venv/bin/python training/evaluate_student.py --answers student_answers.jsonl --label student --out training/data/student_eval.json

**Runtime → Change runtime type → T4 or L4 GPU**, then **Runtime → Run all**. About 10 minutes on an L4.


In [ ]:
import subprocess, sys, importlib

USE_UNSLOTH = True  # the fallback cell sets this to False

def pip(*args):
    cmd = [sys.executable, "-m", "pip", "install", "--quiet", *args]
    print("$", " ".join(cmd))
    result = subprocess.run(cmd, capture_output=True, text=True)
    if result.returncode != 0:
        print(result.stdout[-3000:], result.stderr[-3000:])
        raise RuntimeError("pip failed -- run the fallback cell below instead")

# unsloth pins its own tested versions of trl / peft / transformers / accelerate /
# bitsandbytes / datasets, so we pin unsloth to a release series and let it choose.
pip("--upgrade", "unsloth==2025.9.*", "unsloth_zoo==2025.9.*")
pip("matplotlib")

print()
print("installed versions:")
for name in ["unsloth", "trl", "peft", "transformers", "datasets", "accelerate", "bitsandbytes", "torch"]:
    try:
        mod = importlib.import_module(name)
        print(f"  {name:<14} {getattr(mod, '__version__', '?')}")
    except Exception as exc:  # noqa: BLE001
        print(f"  {name:<14} NOT IMPORTABLE ({exc})")

## 2. The held-out packets and your Drive

In [ ]:
import json, os, urllib.request
from google.colab import drive

drive.mount("/content/drive", timeout_ms=1800000)  # waits up to 30 minutes for the Drive permission click
WORK_DIR = "/content/drive/MyDrive/news-briefer"
ADAPTER_NAME = "news-briefer-lora"       # stage 1; set to "news-briefer-dpo-lora" to score stage 2
ANSWERS_NAME = "student_answers.jsonl"   # set to "dpo_answers.jsonl" for stage 2
SCORE_BASE = True                        # the untuned model only needs scoring once
ADAPTER_DIR = os.path.join(WORK_DIR, ADAPTER_NAME)
assert os.path.isdir(ADAPTER_DIR), f"adapter not found at {ADAPTER_DIR}: run 01_sft_colab.ipynb first (its save cell writes it)"

VAL_FILE = "sft_val.jsonl"
if not os.path.exists(VAL_FILE):
    urllib.request.urlretrieve(
        "https://github.com/sanjivp2703/adaptive-news-briefing-agent/releases/download/training-data-v7/" + VAL_FILE,
        VAL_FILE,
    )
with open(VAL_FILE, encoding="utf-8") as fh:
    val_rows = [json.loads(line) for line in fh if line.strip()]
print(len(val_rows), "held-out packets")


## 3. Generate

The same system and user messages the student was trained on, rendered through the model's chat template, temperature 0.3, up to 1,200 new tokens. The base model gets the identical input.

In [ ]:
import time, torch
from unsloth import FastLanguageModel

BASE = "unsloth/Qwen2.5-3B-Instruct-bnb-4bit"
MAX_NEW = 1200

def generate_all(model, tokenizer, rows, out_path):
    FastLanguageModel.for_inference(model)
    done = 0
    with open(out_path, "w", encoding="utf-8") as out:
        for row in rows:
            messages = row["messages"][:2]
            ids = tokenizer.apply_chat_template(messages, add_generation_prompt=True, return_tensors="pt").to(model.device)
            t0 = time.time()
            with torch.no_grad():
                gen = model.generate(input_ids=ids, max_new_tokens=MAX_NEW, temperature=0.3, do_sample=True,
                                     top_p=0.95, pad_token_id=tokenizer.pad_token_id or tokenizer.eos_token_id)
            text = tokenizer.decode(gen[0][ids.shape[1]:], skip_special_tokens=True)
            out.write(json.dumps({"id": row["meta"]["id"], "profile": row["meta"].get("profile"), "answer": text,
                                  "seconds": round(time.time() - t0, 1)}) + "\n")
            done += 1
            print(f"  {done}/{len(rows)}  {time.time() - t0:5.1f}s  {row['meta'].get('profile')}")
    print("wrote", out_path)

if SCORE_BASE:
    print("=== untuned base model ===")
    model, tokenizer = FastLanguageModel.from_pretrained(model_name=BASE, max_seq_length=8192, dtype=None, load_in_4bit=True)
    if tokenizer.pad_token is None:
        tokenizer.pad_token = tokenizer.eos_token
    generate_all(model, tokenizer, val_rows, os.path.join(WORK_DIR, "base_answers.jsonl"))
    del model; torch.cuda.empty_cache()

print("=== fine-tuned student (base + adapter) ===")
model, tokenizer = FastLanguageModel.from_pretrained(model_name=ADAPTER_DIR, max_seq_length=8192, dtype=None, load_in_4bit=True)
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token
generate_all(model, tokenizer, val_rows, os.path.join(WORK_DIR, ANSWERS_NAME))


## 4. Done

Both files are in `MyDrive/news-briefer/`. Download them to your Mac and run the two scoring commands at the top of this notebook. If Drive is not synced locally, this cell offers them as browser downloads.

In [ ]:
from google.colab import files
for name in (["base_answers.jsonl"] if SCORE_BASE else []) + [ANSWERS_NAME]:
    path = os.path.join(WORK_DIR, name)
    print(name, os.path.getsize(path) // 1024, "KB")
    files.download(path)
